In [1]:
import torch
import pandas as pd
from unfairness import load_hparams, load_category
from unfairness.dataset import load_kb_bank, load_kb_texts, ToS, make_dataloaders
from unfairness.token import tokenizer
from unfairness.infer import predict_with_rationales
from unfairness.utils.config_loader import load_model_and_tokenizer

In [2]:
df = pd.DataFrame({
    "document_ID": [1,1,2,2,3,3],
    "text": [
        "we may collect your data for analytics",          # A=1
        "cookies improve your experience",                 # A=1
        "we will not share data with third parties",       # A=0
        "third parties might receive aggregated stats",    # A=1
        "unsubscribe anytime via settings",                # A=0
        "personal data processed lawfully",                # A=1
    ],
    "A": [1,1,0,1,0,1]
})

test = pd.read_csv('/nvme/git/contest/hackathon2025/cv_test/torch/fold_1/test.csv')

df["A_targets"] = ["[0,1]","[1]","[]","[0]","[]","[2]"]

ckpt_path = '/nvme/git/contest/hackathon2025/cv_test/torch/fold_1/best-v3.ckpt'
hparams = load_hparams("configs/distributed_model_config.json","configs/training_config.json")
category = load_category("configs/data_loader.json")

lit, tok_trained, kb_texts = load_model_and_tokenizer(
    ckpt_path=ckpt_path,
    category=category,
    max_len=32,
    map_location="cuda" if torch.cuda.is_available() else "cpu"
)

te_ds = ToS(test, category, tok_trained, max_len=32)
_, _, te_loader = make_dataloaders(te_ds, None, te_ds, batch_size=4)

In [ ]:
results = predict_with_rationales(
    lit_module=lit,
    dataloader=te_loader,
    kb_texts=kb_texts,
    threshold=0.5,
    top_k=3,
    return_for_all=False,  
    use_scores=True       
)

for r in results[:5]:
    print(f"prob={r['prob']:.3f} pred={r['pred']} gold={r['gold']}")
    for k, rat in enumerate(r["rationales"], 1):
        print(f"  {k}) idx={rat['idx']}  s={rat['score_raw']:.3f}  w={rat['att_weight']:.3f}  text={rat['text']}")
    print("-"*60)

prob=0.493 pred=0 gold=0.0
------------------------------------------------------------
prob=0.557 pred=1 gold=1.0
  1) idx=0  s=-0.269  w=0.433  text=we collect data for analytics
  2) idx=1  s=-0.336  w=0.417  text=we use cookies to improve experience
  3) idx=2  s=-0.562  w=0.363  text=processing of personal data must be lawful
------------------------------------------------------------
